# Diesta Reconciliation Exercise

Use this notebook to reconcile MGA premium cash received, broker remittances, and premium BDX data. Your output should be reproducible and should make clear what matched, what did not match, and what assumptions you made.

## Deliverables

Include a source-data profile, a bank-to-remittance reconciliation, a remittance-to-BDX reconciliation, a financial summary, exception reason codes, and at least five written exception explanations. Do not force a match where the evidence is weak.

In [12]:
from pathlib import Path
import pandas as pd

BASE = Path.cwd()
DATA_CANDIDATES = (BASE / "data", BASE / "candidate_pack" / "data")
DATA = next(
    path for path in DATA_CANDIDATES
    if (path / "bank_transactions_may_2026.csv").exists()
)
REMITTANCES = DATA / "remittances"
BANK_PATH = DATA / "bank_transactions_may_2026.csv"
BDX_PATH = DATA / "premium_bdx_april_2026.csv"

In [13]:
bank = pd.read_csv(BANK_PATH, dtype=str); bdx = pd.read_csv(BDX_PATH, dtype=str); remittance_files = sorted(REMITTANCES.iterdir()); bank.head(), bdx.head(), [p.name for p in remittance_files]

(  transaction_id                account_name booking_date  value_date  \
 0      BANK-0001  MGA Premium Collection GBP   03/05/2026  2026-05-03   
 1      BANK-0002  MGA Premium Collection GBP   03/05/2026  2026-05-03   
 2      BANK-0003  MGA Premium Collection GBP   04/05/2026  2026-05-04   
 3      BANK-0004  MGA Premium Collection GBP   05/05/2026  2026-05-05   
 4      BANK-0005  MGA Premium Collection GBP   2026-05-06  2026-05-06   
 
   currency credit_amount debit_amount            counterparty_name  \
 0      GBP      32517.50          NaN       APEX RISK PARTNERS LTD   
 1      GBP      49990.00          NaN       NORTHSHORE BROKING LTD   
 2      GBP     150000.00          NaN    PIONEER WHOLESALE BROKERS   
 3      GBP      50000.00          NaN    PIONEER WHOLESALE BROKERS   
 4      GBP      11867.80          NaN  CONTINENTAL COVERHOLDERS SA   
 
   bank_reference                                                narrative  
 0      FP 483920          APEX RISK PARTNERS APR

## Source Data Profile

Load the remittance files and briefly profile the supplied data.

In [14]:
import csv
import warnings
# These workbooks have no default style. Ignore that warning so the preview stays readable.
warnings.filterwarnings(
    "ignore",
    message="Workbook contains no default style",
    module="openpyxl",
)

PREVIEW_ROW_COUNT = 3
# A cover sheet is a short list of labels, not a column table. Show the whole list
# when it is small so totals and payment notes stay visible.
SHORT_LABEL_SHEET = 8


def cell_text(value):
    """Return the cell as text, including leading zeros in identifiers."""
    if value is None:
        return ""
    try:
        if pd.isna(value):
            return ""
    except TypeError:
        pass
    return str(value).strip()


def is_column_label(value):
    """A column label is a word such as 'Policy No', not an amount, date, or reference."""
    if value == "":
        return False
    has_letter = any(character.isalpha() for character in value)
    has_digit = any(character.isdigit() for character in value)
    return has_letter and not has_digit


def find_header_index(rows):
    """Find the row of column names.

    Some brokers put a short banner above the table. The column row is the
    widest row that contains only words.
    """
    header_index = None
    header_width = 1
    for index, row in enumerate(rows):
        labels = [cell for cell in row if is_column_label(cell)]
        other_values = [cell for cell in row if cell != "" and cell not in labels]
        if other_values:
            continue
        if len(labels) > header_width:
            header_width = len(labels)
            header_index = index
    return header_index


def show_preview(filename, location, rows):
    """Print the source, the field names, and the first three data rows."""
    print("=" * 72)
    print("File:", filename)
    if location:
        print("Sheet or table:", location)

    stored_rows = [row for row in rows if any(cell != "" for cell in row)]
    if not stored_rows:
        print("Column or field names: none")
        print("First three data rows: this sheet or table is empty.")
        return

    header_index = find_header_index(rows)
    if header_index is None:
        print("Column or field names: no column header on this sheet or table.")
        rows_to_show = stored_rows
        if len(stored_rows) > SHORT_LABEL_SHEET:
            rows_to_show = stored_rows[:PREVIEW_ROW_COUNT]
        print("Stored rows:")
        for row in rows_to_show:
            visible = [cell for cell in row if cell != ""]
            print(" -", " | ".join(visible))
        print(f"Non-empty rows: {len(stored_rows)}. Showing {len(rows_to_show)}.")
        return

    banner_rows = [row for row in rows[:header_index] if any(cell != "" for cell in row)]
    if banner_rows:
        print("Rows above the column header:")
        for row in banner_rows:
            visible = [cell for cell in row if cell != ""]
            print(" -", " | ".join(visible))

    field_names = list(rows[header_index])
    while field_names and field_names[-1] == "":
        field_names.pop()

    print("Column or field names:")
    print(field_names)

    data_rows = []
    for row in rows[header_index + 1 :]:
        if not any(cell != "" for cell in row):
            continue
        values = list(row[: len(field_names)])
        if len(values) < len(field_names):
            values.extend([""] * (len(field_names) - len(values)))
        data_rows.append(values)
        if len(data_rows) == PREVIEW_ROW_COUNT:
            break

    print("First three data rows:")
    if not data_rows:
        print(" (no data rows under this header)")
        return

    preview = pd.DataFrame(data_rows, columns=field_names)
    print(preview.to_string(index=False, max_cols=None, max_colwidth=None))


def read_csv_rows(path):
    """Read a CSV as text. Some files have a banner, so rows are not all the same width."""
    with path.open(newline="", encoding="utf-8-sig") as handle:
        return [[cell_text(value) for value in row] for row in csv.reader(handle)]


def read_excel_sheets(path):
    """Read every sheet as text. dtype=str keeps identifiers unchanged."""
    workbook = pd.read_excel(
        path,
        sheet_name=None,
        header=None,
        dtype=str,
        engine="openpyxl",
    )
    sheets = []
    for sheet_name, frame in workbook.items():
        rows = []
        for record in frame.itertuples(index=False, name=None):
            rows.append([cell_text(value) for value in record])
        sheets.append((sheet_name, rows))
    return sheets


def preview_pdf(path):
    """Extract PDF tables when a PDF reader is installed. Otherwise name the missing package."""
    try:
        import pdfplumber # type: ignore
    except ImportError:
        print("=" * 72)
        print("File:", path.name)
        print(
            "Missing package: pdfplumber. "
            "pandas and openpyxl do not extract PDF tables, "
            "and pdfplumber is not installed in this project."
        )
        return

    found_table = False
    with pdfplumber.open(path) as document:
        for page_number, page in enumerate(document.pages, start=1):
            for table_number, table in enumerate(page.extract_tables() or [], start=1):
                if not table:
                    continue
                found_table = True
                rows = [
                    [cell_text(value) for value in (raw_row or [])]
                    for raw_row in table
                ]
                location = f"page {page_number}, table {table_number}"
                show_preview(path.name, location, rows)

    if not found_table:
        print("=" * 72)
        print("File:", path.name)
        print("No extractable table in this PDF.")


# Build the file list again on each run. This cell only reads the files.
remittance_files = sorted(
    path
    for path in REMITTANCES.iterdir()
    if path.is_file() and not path.name.startswith(".")
)

if not remittance_files:
    print("No files found in", REMITTANCES)

for remittance_path in remittance_files:
    suffix = remittance_path.suffix.lower()
    try:
        if suffix == ".csv":
            show_preview(remittance_path.name, None, read_csv_rows(remittance_path))
        elif suffix in {".xlsx", ".xlsm"}:
            check = read_excel_sheets(remittance_path)
            for sheet_name, sheet_rows in read_excel_sheets(remittance_path):
                show_preview(remittance_path.name, sheet_name, sheet_rows)
        elif suffix == ".xls":
            print("=" * 72)
            print("File:", remittance_path.name)
            print("Missing package: xlrd. openpyxl reads .xlsx files, but not older .xls files.")
#         elif suffix == ".pdf":
#             preview_pdf(remittance_path)
        else:
            print("=" * 72)
            print("File:", remittance_path.name)
            print(f"Unsupported file type '{suffix or '(none)'}' for this inspection.")
    except ImportError as error:
        print("=" * 72)
        print("File:", remittance_path.name)
        print("Missing package:", error)
    except Exception as error:
        print("=" * 72)
        print("File:", remittance_path.name)
        print("Could not read this file:", error)


File: apex_risk_partners_remittance_apr_2026.csv
Rows above the column header:
 - Broker | Apex Risk Partners Ltd
 - Statement ID | APX-APR-2026
 - Period | April 2026
Column or field names:
['Broker', 'Statement ID', 'Policy No', 'Insured', 'Transaction Type', 'Accounting Period', 'Gross Premium', 'Commission %', 'Commission Amount', 'IPT', 'Net Settlement', 'Status / Notes']
First three data rows:
                Broker Statement ID         Policy No                       Insured Transaction Type Accounting Period Gross Premium Commission % Commission Amount          IPT Net Settlement Status / Notes
Apex Risk Partners Ltd APX-APR-2026         2026/0001 Sterling Construction Limited               NB           2026-04 GBP 10,000.00          15%        (1,500.00) GBP 1,200.00   GBP 8,500.00               
Apex Risk Partners Ltd APX-APR-2026           POL0002         A.B.C. Technology Ltd               NB           2026-04  GBP 7,500.00          15%        (1,125.00)   GBP 900.00   GBP 

### Loading decisions

Bank and premium BDX files are ordinary header-on-row-1 CSVs. Apex puts a short banner above its detail header, so that header is detected rather than assumed to be row 1. Excel workbooks are read sheet by sheet: Northshore `Cover` and Harbour `Summary` are label lists, kept as controls. A row whose first value is `TOTAL` is a statement total, not a policy line.

Every value stays text. Each detail row keeps `source_file`, `source_sheet` (blank for CSV), and a 1-based `source_row`. Duplicate lines, blank fields, and mixed amount or date formats are left in the detail tables and noted in the profile.

In [ ]:
PROVENANCE = ("source_file", "source_sheet", "source_row")

KEY_FIELDS = {
    "bank_transactions_may_2026.csv": [
        "transaction_id", "booking_date", "currency", "credit_amount", "counterparty_name", "narrative",
    ],
    "premium_bdx_april_2026.csv": [
        "bdx_record_id", "broker_name", "policy_reference", "original_currency",
        "net_due_to_mga_original_ccy", "settlement_amount_gbp",
    ],
    "apex_risk_partners_remittance_apr_2026.csv": [
        "Policy No", "Insured", "Gross Premium", "Net Settlement",
    ],
    "pioneer_wholesale_statement_may_2026.csv": [
        "Policy Reference", "Client", "Currency", "Amount Due",
    ],
    "continental_coverholders_apr_2026.xlsx": [
        "Policy", "Original CCY", "Net Due Original CCY", "GBP Settlement Amount",
    ],
    "harbour_specialty_remittance_apr_2026.xlsx": [
        "Policy / Risk Ref", "Insured Name", "Gross", "Net Due",
    ],
    "northshore_broking_remittance_apr_2026.xlsx": [
        "Contract No", "UW Ref", "Currency", "Amount Due MGA",
    ],
}


def trim_field_names(row):
    names = list(row)
    while names and names[-1] == "":
        names.pop()
    used = {}
    cleaned = []
    for index, name in enumerate(names):
        label = name or f"unnamed_{index + 1}"
        used[label] = used.get(label, 0) + 1
        if used[label] > 1:
            label = f"{label}_{used[label]}"
        cleaned.append(label)
    return cleaned


def control_record(source_file, source_sheet, source_row, control_type, label, value):
    return {
        "source_file": source_file,
        "source_sheet": source_sheet,
        "source_row": source_row,
        "control_type": control_type,
        "label": label,
        "value": value,
    }


def split_detail_and_controls(rows, source_file, source_sheet=""):
    """Separate a policy table from banners, cover sheets, and statement totals."""
    controls = []
    header_index = find_header_index(rows)
    if header_index is None:
        for row_number, row in enumerate(rows, start=1):
            visible = [cell for cell in row if cell != ""]
            if not visible:
                continue
            value = " | ".join(visible[1:]) if len(visible) > 1 else ""
            controls.append(control_record(
                source_file, source_sheet, row_number, "cover_sheet", visible[0], value,
            ))
        return pd.DataFrame(), controls

    for row_number, row in enumerate(rows[:header_index], start=1):
        visible = [cell for cell in row if cell != ""]
        if not visible:
            continue
        value = " | ".join(visible[1:]) if len(visible) > 1 else ""
        controls.append(control_record(
            source_file, source_sheet, row_number, "banner", visible[0], value,
        ))

    field_names = trim_field_names(rows[header_index])
    detail_rows = []
    for row_number, row in enumerate(rows[header_index + 1:], start=header_index + 2):
        if not any(cell != "" for cell in row):
            continue
        values = list(row[:len(field_names)])
        if len(values) < len(field_names):
            values.extend([""] * (len(field_names) - len(values)))
        first_value = next((cell for cell in values if cell != ""), "")
        if first_value.casefold() == "total":
            amount = next((cell for cell in reversed(values) if cell and cell.casefold() != "total"), "")
            controls.append(control_record(
                source_file, source_sheet, row_number, "statement_total", "TOTAL", amount,
            ))
            continue
        record = dict(zip(field_names, values))
        record["source_file"] = source_file
        record["source_sheet"] = source_sheet
        record["source_row"] = row_number
        detail_rows.append(record)

    return pd.DataFrame(detail_rows), controls


def load_table(path, sheet_name="", rows=None):
    if rows is None:
        rows = read_csv_rows(path)
    return split_detail_and_controls(rows, path.name, sheet_name)


bank_transactions, bank_controls = load_table(BANK_PATH)
premium_bdx, bdx_controls = load_table(BDX_PATH)

remittance_details = {}
remittance_controls = []
for remittance_path in remittance_files:
    suffix = remittance_path.suffix.lower()
    if suffix == ".csv":
        detail, controls = load_table(remittance_path)
        remittance_details[remittance_path.name] = detail
        remittance_controls.extend(controls)
    elif suffix in {".xlsx", ".xlsm"}:
        for sheet_name, sheet_rows in read_excel_sheets(remittance_path):
            detail, controls = load_table(remittance_path, sheet_name, sheet_rows)
            if not detail.empty:
                remittance_details[f"{remittance_path.name} | {sheet_name}"] = detail
            remittance_controls.extend(controls)

source_controls = pd.DataFrame(bank_controls + bdx_controls + remittance_controls)
loaded_sources = {
    "bank_transactions": bank_transactions,
    "premium_bdx": premium_bdx,
    "remittance_details": remittance_details,
    "source_controls": source_controls,
}


def missing_key_fields(frame, filename):
    notes = []
    for column in KEY_FIELDS.get(filename, []):
        if column not in frame.columns:
            notes.append(f"{column}: column absent")
            continue
        missing = int(frame[column].eq("").sum())
        if missing:
            notes.append(f"{column}: {missing}")
    return "; ".join(notes) if notes else "none"


CURRENCY_CODES = ("GBP", "EUR", "USD")


def currency_codes(value):
    """Currency codes stored as the whole value or as an amount prefix such as 'GBP 1,200.00'."""
    upper = value.upper()
    found = []
    for code in CURRENCY_CODES:
        if upper == code or upper.startswith(code + " "):
            found.append(code)
    return found


def currencies_in(frame):
    found = set()
    for column in frame.columns:
        if column in PROVENANCE:
            continue
        for value in frame[column]:
            if value:
                found.update(currency_codes(value))
    return found


def currencies_text(frame, control_text=""):
    found = set(currencies_in(frame))
    for code in CURRENCY_CODES:
        if code in control_text:
            found.add(code)
    return ", ".join(sorted(found))


def quality_notes(frame):
    notes = []
    business_columns = [column for column in frame.columns if column not in PROVENANCE]
    duplicate_count = int(frame.duplicated(subset=business_columns, keep=False).sum())
    if duplicate_count:
        notes.append(f"{duplicate_count} detail rows are exact duplicates of another row")
    for column in business_columns:
        values = [value for value in frame[column] if value]
        has_currency_prefix = any(value[:3].upper() in {"GBP", "EUR", "USD"} for value in values)
        has_bare_number = any(value[:1].isdigit() or value[:1] in "-(" for value in values)
        if has_currency_prefix and has_bare_number:
            notes.append(f"{column} mixes currency prefixes and bare numbers")
        date_like = [value for value in values if "/" in value or (len(value) >= 10 and value[4:5] == "-")]
        slash_dates = sum("/" in value for value in date_like)
        iso_dates = sum(len(value) >= 10 and value[4:5] == "-" for value in date_like)
        if slash_dates and iso_dates:
            notes.append(f"{column} mixes slash dates and ISO dates")
    blank_columns = [
        column for column in business_columns
        if frame[column].eq("").any()
    ]
    if blank_columns:
        notes.append("blank values in " + ", ".join(blank_columns))
    return "; ".join(notes) if notes else "none noted"


def controls_for(filename, sheet_name=""):
    if source_controls.empty:
        return ""
    matched = source_controls[source_controls["source_file"].eq(filename)]
    if sheet_name:
        matched = matched[matched["source_sheet"].eq(sheet_name)]
    elif filename.endswith(".csv"):
        matched = matched[matched["source_sheet"].eq("")]
    if matched.empty:
        return ""
    return "; ".join(
        f"{row.control_type}: {row.label}={row.value} (row {row.source_row})"
        for row in matched.itertuples(index=False)
    )


profile_rows = []
for label, frame in (
    ("bank_transactions", bank_transactions),
    ("premium_bdx", premium_bdx),
):
    filename = frame["source_file"].iloc[0]
    profile_rows.append({
        "dataset": label,
        "source_file": filename,
        "source_sheet": "",
        "detail_rows": len(frame),
        "columns": ", ".join(column for column in frame.columns if column not in PROVENANCE),
        "missing_key_fields": missing_key_fields(frame, filename),
        "currencies": currencies_text(frame, controls_for(filename)),
        "controls": controls_for(filename),
        "observations": quality_notes(frame),
    })

for dataset_name, frame in remittance_details.items():
    filename = frame["source_file"].iloc[0]
    sheet_name = frame["source_sheet"].iloc[0]
    control_text = controls_for(filename, sheet_name)
    profile_rows.append({
        "dataset": dataset_name,
        "source_file": filename,
        "source_sheet": sheet_name,
        "detail_rows": len(frame),
        "columns": ", ".join(column for column in frame.columns if column not in PROVENANCE),
        "missing_key_fields": missing_key_fields(frame, filename),
        "currencies": currencies_text(frame, control_text),
        "controls": control_text,
        "observations": quality_notes(frame),
    })

# Cover sheets have no detail rows, so add them from the control table.
cover_sheets = source_controls[source_controls["control_type"].eq("cover_sheet")]
for (filename, sheet_name), group in cover_sheets.groupby(["source_file", "source_sheet"], sort=False):
    profile_rows.append({
        "dataset": f"{filename} | {sheet_name}",
        "source_file": filename,
        "source_sheet": sheet_name,
        "detail_rows": 0,
        "columns": "",
        "missing_key_fields": "",
        "currencies": ", ".join(
            code for code in CURRENCY_CODES
            if code in " ".join(group["value"].astype(str))
        ),
        "controls": "; ".join(
            f"cover_sheet: {row.label}={row.value} (row {row.source_row})"
            for row in group.itertuples(index=False)
        ),
        "observations": "cover or summary sheet; no policy-level rows",
    })

source_profile = pd.DataFrame(profile_rows)
pd.set_option("display.max_colwidth", None)
# pd.set_option("display.max_columns", None)
# pd.set_option("display.width", None)
display(source_profile)
source_controls

,dataset,source_file,source_sheet,detail_rows,columns,missing_key_fields,currencies,controls,observations
0,bank_transactions,bank_transactions_may_2026.csv,,7,"transaction_id, account_name, booking_date, value_date, currency, credit_amount, debit_amount, counterparty_name, bank_reference, narrative",credit_amount: 1,GBP,,"booking_date mixes slash dates and ISO dates; blank values in credit_amount, debit_amount"
1,premium_bdx,premium_bdx_april_2026.csv,,22,"bdx_record_id, broker_name, broker_id, policy_reference, risk_reference, umr, insured_name, inception_date, transaction_type, transactio...",none,"EUR, GBP",,blank values in bdx_notes
2,apex_risk_partners_remittance_apr_2026.csv,apex_risk_partners_remittance_apr_2026.csv,,7,"Broker, Statement ID, Policy No, Insured, Transaction Type, Accounting Period, Gross Premium, Commission %, Commission Amount, IPT, Net ...",none,GBP,banner: Broker=Apex Risk Partners Ltd (row 1); banner: Statement ID=APX-APR-2026 (row 2); banner: Period=April 2026 (row 3); statement_t...,2 detail rows are exact duplicates of another row; Gross Premium mixes currency prefixes and bare numbers; IPT mixes currency prefixes a...
3,continental_coverholders_apr_2026.xlsx | EUR Settlement,continental_coverholders_apr_2026.xlsx,EUR Settlement,3,"Policy, Insured, Txn Type, Original CCY, Net Due Original CCY, FX Rate Used, GBP Settlement Amount, Notes",none,"EUR, GBP",banner: Broker=Continental Coverholders SA (row 1); banner: Statement Ref=CON-APR-2026 (row 2); banner: Settlement CCY=GBP (row 3); stat...,blank values in Notes
4,harbour_specialty_remittance_apr_2026.xlsx | Items,harbour_specialty_remittance_apr_2026.xlsx,Items,3,"Line ID, Policy / Risk Ref, Insured Name, Inception, Txn Type, Gross, Commission, Net Due, Comment",Policy / Risk Ref: 1,,statement_total: TOTAL=36500 (row 5),"blank values in Policy / Risk Ref, Comment"
5,northshore_broking_remittance_apr_2026.xlsx | Remittance,northshore_broking_remittance_apr_2026.xlsx,Remittance,4,"Contract No, Assured, UW Ref, Transaction Date, Currency, Gross Amount, Brokerage %, Brokerage Amount, Tax/IPT, Amount Due MGA, Comments",none,GBP,statement_total: TOTAL=50000 (row 6),blank values in Comments
6,pioneer_wholesale_statement_may_2026.csv,pioneer_wholesale_statement_may_2026.csv,,4,"Broker, Statement Ref, Payment Ref, Policy Reference, Client, Txn Date, Currency, Gross, Brokerage, IPT, Amount Due, Comment",none,GBP,statement_total: TOTAL=200000.00 (row 6),Txn Date mixes slash dates and ISO dates; blank values in Comment
7,harbour_specialty_remittance_apr_2026.xlsx | Summary,harbour_specialty_remittance_apr_2026.xlsx,Summary,0,,,GBP,cover_sheet: Harbour Specialty Ltd= (row 1); cover_sheet: Statement Ref=HRB-APR-2026 (row 2); cover_sheet: Period=April 2026 (row 3); co...,cover or summary sheet; no policy-level rows
8,northshore_broking_remittance_apr_2026.xlsx | Cover,northshore_broking_remittance_apr_2026.xlsx,Cover,0,,,GBP,cover_sheet: Northshore Broking Ltd= (row 1); cover_sheet: Remittance ID=REM-0426-NSB (row 2); cover_sheet: Period=April 2026 (row 3); c...,cover or summary sheet; no policy-level rows


,source_file,source_sheet,source_row,control_type,label,value
0,apex_risk_partners_remittance_apr_2026.csv,,1,banner,Broker,Apex Risk Partners Ltd
1,apex_risk_partners_remittance_apr_2026.csv,,2,banner,Statement ID,APX-APR-2026
2,apex_risk_partners_remittance_apr_2026.csv,,3,banner,Period,April 2026
3,apex_risk_partners_remittance_apr_2026.csv,,13,statement_total,TOTAL,"GBP 32,517.50"
4,continental_coverholders_apr_2026.xlsx,EUR Settlement,1,banner,Broker,Continental Coverholders SA
5,continental_coverholders_apr_2026.xlsx,EUR Settlement,2,banner,Statement Ref,CON-APR-2026
6,continental_coverholders_apr_2026.xlsx,EUR Settlement,3,banner,Settlement CCY,GBP
7,continental_coverholders_apr_2026.xlsx,EUR Settlement,9,statement_total,TOTAL,11868
8,harbour_specialty_remittance_apr_2026.xlsx,Summary,1,cover_sheet,Harbour Specialty Ltd,
9,harbour_specialty_remittance_apr_2026.xlsx,Summary,2,cover_sheet,Statement Ref,HRB-APR-2026


### What the profile shows

All seven files load. Bank has 7 cash lines (one of them a fee debit with no credit). Premium BDX has 22 policy records and no statement total. Each remittance detail sheet loads separately from its banner or cover sheet.

Apex’s banner (`APX-APR-2026`) sits above 7 detail lines, including two identical `POL-2026-0006` rows, and a statement total of `GBP 32,517.50`. Amounts on that file mix `GBP` prefixes, parentheses, and bare numbers, and policy numbers are not in one format. Pioneer has 4 lines, a `200,000.00` total, and both ISO and slash dates. Northshore’s cover states `GBP 50,000.00` and a CHAPS payment note; its remittance sheet has 4 lines totalling `50000`. Harbour’s summary states `GBP 36,500.00`; the items sheet has 3 lines totalling `36500`. Line `HRB-1` quotes `H-0301` rather than the BDX policy reference, and line `HRB-2` has no policy reference. Continental’s sheet is labelled settlement currency GBP, with 3 EUR lines and a GBP total of `11868`.

These totals and the repeated Thames Logistics line are used in the two reconciliations below. Bank cash is linked to statements, and remittance lines are linked to BDX, in separate stages.


## Bank To Remittance Reconciliation

Cash received is matched to each broker statement using the counterparty name and any statement or payment reference in the bank reference or narrative. The amount and currency are checks after that link is made. A credit is not assigned to a broker on amount alone.

**Tolerance.** Amounts are compared to the penny. A link is called `matched` only when the absolute variance is at most **GBP 0.01**. Any larger difference stays on the row as the actual variance. Nothing is rounded to force a match.

**Assumptions.**

- Only rows with a bank credit are premium receipts. `BANK-0007`, the GBP 30 account-charge debit, is left out.
- One statement can be paid by more than one credit. Those credits are listed individually and summed against the statement total.
- The statement total is the cover or banner stated total when the file has one, otherwise the `TOTAL` row. Continental is compared in its stated settlement currency (GBP), not the EUR original amounts.
- A bank credit with no broker name and no statement reference in common with a remittance stays unresolved.

In [16]:
import re

ROUNDING_TOLERANCE = 0.01  # GBP; differences above one penny are reported

NAME_STOPWORDS = {
    "LTD", "LIMITED", "PLC", "SA", "LLP", "INC",
    "BROKER", "BROKERS", "BROKING", "WHOLESALE", "PARTNERS",
    "SPECIALTY", "COVERHOLDERS", "RISK", "SERVICES", "AND", "THE",
}
REF_LABELS = {"statement id", "statement ref", "remittance id"}
TOTAL_LABELS = {"stated total", "statement total"}
BROKER_LABELS = {"broker"}
CCY_LABELS = {"settlement ccy", "currency"}


def parse_amount(text):
    """Parse a stated amount, including a currency prefix and thousands separators."""
    if text is None or str(text).strip() == "":
        return None
    raw = str(text).strip()
    negative = raw.startswith("(") and raw.endswith(")")
    cleaned = raw.strip("()")
    cleaned = re.sub(r"\b[A-Z]{3}\b", "", cleaned)
    cleaned = cleaned.replace(",", "").strip()
    if cleaned in {"", "-"}:
        return None
    value = float(cleaned)
    return -value if negative else value


def currency_of(text):
    if text is None:
        return ""
    match = re.search(r"\b(GBP|EUR|USD)\b", str(text).upper())
    return match.group(1) if match else ""


def name_tokens(broker):
    tokens = re.findall(r"[A-Z0-9]+", str(broker).upper())
    return [token for token in tokens if token not in NAME_STOPWORDS and len(token) >= 4]


def text_has_token(text, token):
    return re.search(rf"\b{re.escape(token)}\b", text) is not None


def file_controls(filename):
    return source_controls[source_controls["source_file"].eq(filename)]


def statement_for(filename, frames):
    controls = file_controls(filename)
    detail = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

    broker = ""
    statement_ref = ""
    currency = ""
    stated_total = None
    stated_total_text = ""
    total_file = filename
    total_sheet = ""
    total_row = ""

    for row in controls.itertuples(index=False):
        label = str(row.label).strip()
        label_key = label.casefold()
        value = str(row.value).strip()
        if label_key in BROKER_LABELS and value:
            broker = value
        elif label_key in REF_LABELS and value and not statement_ref:
            statement_ref = value
        elif label_key in CCY_LABELS and value:
            currency = currency_of(value) or value.upper()
        elif label_key in TOTAL_LABELS and value:
            stated_total = parse_amount(value)
            stated_total_text = value
            currency = currency or currency_of(value)
            total_sheet = row.source_sheet
            total_row = row.source_row
        elif row.control_type == "statement_total" and stated_total is None:
            stated_total = parse_amount(value)
            stated_total_text = value
            currency = currency or currency_of(value)
            total_sheet = row.source_sheet
            total_row = row.source_row
        elif row.control_type == "cover_sheet" and not value and not broker:
            broker = label

    if not detail.empty:
        columns = {column.casefold(): column for column in detail.columns}
        if not broker and "broker" in columns:
            broker = next(value for value in detail[columns["broker"]] if value)
        if not statement_ref:
            for key in ("statement id", "statement ref", "remittance id"):
                if key in columns and detail[columns[key]].ne("").any():
                    statement_ref = next(value for value in detail[columns[key]] if value)
                    break
        if not currency and "currency" in columns:
            codes = sorted({value for value in detail[columns["currency"]] if currency_of(value) or len(value) == 3})
            if len(codes) == 1:
                currency = currency_of(codes[0]) or codes[0].upper()

    refs = []
    for ref in [statement_ref]:
        if ref and ref not in refs:
            refs.append(ref)
    if not detail.empty:
        for key in ("payment ref", "statement id", "statement ref", "remittance id"):
            column = {name.casefold(): name for name in detail.columns}.get(key)
            if column:
                for value in detail[column]:
                    if value and value not in refs:
                        refs.append(value)

    return {
        "broker": broker,
        "statement_reference": statement_ref,
        "match_refs": refs,
        "name_tokens": name_tokens(broker),
        "currency": currency or "GBP",
        "statement_total": stated_total,
        "statement_total_text": stated_total_text,
        "source_file": total_file,
        "source_sheet": total_sheet,
        "source_row": total_row,
    }


frames_by_file = {}
for dataset_name, frame in remittance_details.items():
    filename = dataset_name.split(" | ", 1)[0]
    frames_by_file.setdefault(filename, []).append(frame)

statements = [statement_for(filename, frames) for filename, frames in frames_by_file.items()]
statements = [row for row in statements if row["statement_reference"] or row["statement_total"] is not None]

bank = bank_transactions.copy()
bank["credit"] = bank["credit_amount"].map(parse_amount)
bank["debit"] = bank["debit_amount"].map(parse_amount)
credits = bank[bank["credit"].notna() & (bank["credit"] > 0)].copy()
excluded_debits = bank[bank["debit"].notna() & (bank["debit"] > 0) & bank["credit"].isna()]


def bank_blob(row):
    return " ".join(str(getattr(row, column)) for column in ("counterparty_name", "bank_reference", "narrative")).upper()


def matching_statements(row):
    blob = bank_blob(row)
    hits = []
    for statement in statements:
        ref_hits = [ref for ref in statement["match_refs"] if ref.upper() in blob]
        token_hits = [token for token in statement["name_tokens"] if text_has_token(blob, token)]
        if ref_hits or token_hits:
            hits.append((statement, ref_hits, token_hits))
    return hits


assignments = []
unresolved_credits = []
for row in credits.itertuples(index=False):
    hits = matching_statements(row)
    receipt = {
        "transaction_id": row.transaction_id,
        "receipt_date": row.value_date,
        "currency": row.currency,
        "amount": row.credit,
        "counterparty_name": row.counterparty_name,
        "bank_reference": row.bank_reference,
        "narrative": row.narrative,
        "source_file": row.source_file,
        "source_row": row.source_row,
    }
    if len(hits) == 1:
        statement, ref_hits, token_hits = hits[0]
        receipt["statement_reference"] = statement["statement_reference"]
        receipt["link"] = "; ".join(
            ([f"reference {', '.join(ref_hits)}"] if ref_hits else [])
            + ([f"name {', '.join(token_hits)}"] if token_hits else [])
        )
        if row.currency != statement["currency"]:
            receipt["link"] += f"; currency {row.currency} vs statement {statement['currency']}"
        assignments.append(receipt)
    else:
        receipt["statement_reference"] = ""
        if len(hits) > 1:
            receipt["link"] = "more than one statement shares this name or reference; left unmatched"
        else:
            receipt["link"] = "no broker name or statement reference in the bank fields"
        unresolved_credits.append(receipt)

assigned = pd.DataFrame(assignments)
results = []

for statement in statements:
    linked = (
        assigned[assigned["statement_reference"].eq(statement["statement_reference"])]
        if not assigned.empty
        else assigned
    )
    total = statement["statement_total"]
    ids = ", ".join(linked["transaction_id"]) if not linked.empty else ""
    dates = ", ".join(linked["receipt_date"]) if not linked.empty else ""
    narratives = " | ".join(linked["narrative"]) if not linked.empty else ""
    links = "; ".join(dict.fromkeys(linked["link"])) if not linked.empty else ""
    receipt_currencies = sorted(set(linked["currency"])) if not linked.empty else []
    same_currency = linked[linked["currency"].eq(statement["currency"])] if not linked.empty else linked
    other_currency = linked[linked["currency"].ne(statement["currency"])] if not linked.empty else linked

    received = None
    variance = None
    if linked.empty:
        received = 0.0
        status = "no identified bank receipt"
        reason_code = "NO_IDENTIFIED_RECEIPT"
        reason = "No bank credit names this broker or quotes this statement reference."
    elif not other_currency.empty:
        # Do not add a receipt in another currency to this statement total.
        status = "currency mismatch"
        reason_code = "CURRENCY_MISMATCH"
        reason = (
            f"{links}. Receipt currency {', '.join(receipt_currencies)} "
            f"is not the statement currency {statement['currency']}. Amounts were not compared."
        )
    elif total is None or same_currency["amount"].isna().any():
        status = "missing amount"
        reason_code = "MISSING_AMOUNT"
        reason = f"{links}. A statement total or receipt amount is missing, so no variance was calculated."
    else:
        received = round(float(same_currency["amount"].sum()), 2)
        variance = round(received - total, 2)
        if abs(variance) <= ROUNDING_TOLERANCE:
            status = "matched"
            reason_code = "MATCHED"
            reason = f"{links}. Combined receipts {received:,.2f} equal the statement total."
        else:
            status = "amount difference"
            reason_code = "AMOUNT_VARIANCE"
            reason = (
                f"{links}. Bank received {received:,.2f} against statement total {total:,.2f} "
                f"(variance {variance:,.2f} {statement['currency']})."
            )
            if narratives:
                reason += f" Bank narrative: {narratives}."

    source = statement["source_file"]
    if statement["source_sheet"]:
        source = f"{source} | {statement['source_sheet']}"

    results.append({
        "broker": statement["broker"],
        "statement_reference": statement["statement_reference"],
        "bank_transaction_ids": ids,
        "receipt_dates": dates,
        "statement_total": total,
        "currency": statement["currency"],
        "bank_amount_received": received,
        "variance": variance,
        "status": status,
        "reason_code": reason_code,
        "reason": reason,
        "source_file_sheet": source,
        "source_row": statement["source_row"],
        "bank_narratives": narratives,
    })

for receipt in unresolved_credits:
    results.append({
        "broker": receipt["counterparty_name"],
        "statement_reference": "",
        "bank_transaction_ids": receipt["transaction_id"],
        "receipt_dates": receipt["receipt_date"],
        "statement_total": None,
        "currency": receipt["currency"],
        "bank_amount_received": round(receipt["amount"], 2),
        "variance": None,
        "status": "unmatched bank credit",
        "reason_code": "UNMATCHED_BANK_CREDIT",
        "reason": f"{receipt['link']}. Narrative: {receipt['narrative']}.",
        "source_file_sheet": receipt["source_file"],
        "source_row": receipt["source_row"],
        "bank_narratives": receipt["narrative"],
    })

bank_to_remittance = pd.DataFrame(results)
bank_receipt_lines = pd.DataFrame(assignments + unresolved_credits)

pd.set_option("display.max_colwidth", 160)
pd.set_option("display.max_rows", 50)
display(bank_to_remittance)
display(bank_receipt_lines)
print(
    f"Rounding tolerance: {ROUNDING_TOLERANCE:.2f}. "
    f"Debits excluded from premium receipts: "
    + ", ".join(
        f"{row.transaction_id} {row.currency} {row.debit:.2f} ({row.narrative})"
        for row in excluded_debits.itertuples(index=False)
    )
)

,broker,statement_reference,bank_transaction_ids,receipt_dates,statement_total,currency,bank_amount_received,variance,status,reason_code,reason,source_file_sheet,source_row,bank_narratives
0,Apex Risk Partners Ltd,APX-APR-2026,BANK-0001,2026-05-03,32517.5,GBP,32517.5,0.0,matched,MATCHED,"reference APX-APR-2026; name APEX. Combined receipts 32,517.50 equal the statement total.",apex_risk_partners_remittance_apr_2026.csv,13,APEX RISK PARTNERS APRIL STATEMENT APX-APR-2026
1,Continental Coverholders SA,CON-APR-2026,BANK-0005,2026-05-06,11868.0,GBP,11867.8,-0.2,amount difference,AMOUNT_VARIANCE,"name CONTINENTAL. Bank received 11,867.80 against statement total 11,868.00 (variance -0.20 GBP). Bank narrative: CONTINENTAL COVERHOLDERS APRIL SETTLEMENT ...",continental_coverholders_apr_2026.xlsx | EUR Settlement,9,CONTINENTAL COVERHOLDERS APRIL SETTLEMENT EUR FX 0.8600
2,Harbour Specialty Ltd,HRB-APR-2026,,,36500.0,GBP,0.0,NaN,no identified bank receipt,NO_IDENTIFIED_RECEIPT,No bank credit names this broker or quotes this statement reference.,harbour_specialty_remittance_apr_2026.xlsx | Summary,4,
3,Northshore Broking Ltd,REM-0426-NSB,BANK-0002,2026-05-03,50000.0,GBP,49990.0,-10.0,amount difference,AMOUNT_VARIANCE,"reference REM-0426-NSB; name NORTHSHORE. Bank received 49,990.00 against statement total 50,000.00 (variance -10.00 GBP). Bank narrative: NORTHSHORE BROKING...",northshore_broking_remittance_apr_2026.xlsx | Cover,4,NORTHSHORE BROKING LTD REM-0426-NSB OUR CHG GBP10 NET
4,Pioneer Wholesale Brokers,PIO-MAY26,"BANK-0003, BANK-0004","2026-05-04, 2026-05-05",200000.0,GBP,200000.0,0.0,matched,MATCHED,"reference PIO-MAY26; name PIONEER. Combined receipts 200,000.00 equal the statement total.",pioneer_wholesale_statement_may_2026.csv,6,PIONEER WHOLESALE REM PIO-MAY26 A | PIONEER WHOLESALE REM PIO-MAY26 B
5,UNKNOWN BROKER SERVICES,,BANK-0006,2026-05-07,NaN,GBP,28750.0,NaN,unmatched bank credit,UNMATCHED_BANK_CREDIT,no broker name or statement reference in the bank fields. Narrative: UBS CLIENT MONEY MAY PREMIUMS.,bank_transactions_may_2026.csv,7,UBS CLIENT MONEY MAY PREMIUMS


,transaction_id,receipt_date,currency,amount,counterparty_name,bank_reference,narrative,source_file,source_row,statement_reference,link
0,BANK-0001,2026-05-03,GBP,32517.5,APEX RISK PARTNERS LTD,FP 483920,APEX RISK PARTNERS APRIL STATEMENT APX-APR-2026,bank_transactions_may_2026.csv,2,APX-APR-2026,reference APX-APR-2026; name APEX
1,BANK-0002,2026-05-03,GBP,49990.0,NORTHSHORE BROKING LTD,CHAPS 809144,NORTHSHORE BROKING LTD REM-0426-NSB OUR CHG GBP10 NET,bank_transactions_may_2026.csv,3,REM-0426-NSB,reference REM-0426-NSB; name NORTHSHORE
2,BANK-0003,2026-05-04,GBP,150000.0,PIONEER WHOLESALE BROKERS,CHAPS 881204,PIONEER WHOLESALE REM PIO-MAY26 A,bank_transactions_may_2026.csv,4,PIO-MAY26,reference PIO-MAY26; name PIONEER
3,BANK-0004,2026-05-05,GBP,50000.0,PIONEER WHOLESALE BROKERS,CHAPS 881205,PIONEER WHOLESALE REM PIO-MAY26 B,bank_transactions_may_2026.csv,5,PIO-MAY26,reference PIO-MAY26; name PIONEER
4,BANK-0005,2026-05-06,GBP,11867.8,CONTINENTAL COVERHOLDERS SA,SEPA 372994,CONTINENTAL COVERHOLDERS APRIL SETTLEMENT EUR FX 0.8600,bank_transactions_may_2026.csv,6,CON-APR-2026,name CONTINENTAL
5,BANK-0006,2026-05-07,GBP,28750.0,UNKNOWN BROKER SERVICES,FP 992310,UBS CLIENT MONEY MAY PREMIUMS,bank_transactions_may_2026.csv,7,,no broker name or statement reference in the bank fields


Rounding tolerance: 0.01. Debits excluded from premium receipts: BANK-0007 GBP 30.00 (MONTHLY ACCOUNT CHARGES)


## Remittance To Premium BDX Reconciliation

Each remittance detail line is compared with the premium BDX on broker and policy reference. Policy references are normalized only for the comparison: case, spaces, hyphens, and slashes are ignored, a trailing `CAN` or `ENDT` suffix is dropped, and a bare `2026/0001` or `POL0002` form is read as `POL-2026-000n`. The original reference is kept on the output.

Insured name, transaction type, currency, and net amount are checks after a unique policy link. Amounts are compared in the same currency. Continental uses its GBP settlement amount against the BDX GBP settlement amount. A link is `matched` only when the absolute variance is at most **GBP 0.01**; the actual variance is still shown.

A BDX record is used at most once. A repeated remittance line does not take that record again. A line with no policy reference is left unresolved when more than one BDX record could fit.

A remittance reference that is not the BDX policy reference can still be linked when, for the same broker, its numeric tail is the trailing part of both that record’s policy reference and its risk reference, and the insured name and same-currency amount agree. `H-0301` is assessed that way. If more than one record could fit, the line stays unresolved and those records are listed as unconfirmed candidates.

A later line is a duplicate only when the broker, policy reference, insured name, transaction type, currency, and amount repeat an earlier line. A repeated policy reference with different business fields is not treated as a duplicate.

Amounts are never subtracted or added across currencies. A currency difference is `currency mismatch`, and a blank amount is `missing amount`. Neither is given a variance.


In [17]:
import re

PENNY_TOLERANCE = 0.01

TXN_GROUPS = {
    "NB": "new business",
    "NEW BUSINESS": "new business",
    "CAN": "cancellation",
    "CANCELLATION": "cancellation",
    "MTA": "adjustment",
    "END": "adjustment",
    "ENDORSEMENT": "adjustment",
}


def parse_money(text):
    if text is None or str(text).strip() == "":
        return None
    raw = str(text).strip()
    negative = raw.startswith("(") and raw.endswith(")")
    cleaned = re.sub(r"\b[A-Z]{3}\b", "", raw.strip("()"))
    cleaned = cleaned.replace(",", "").strip()
    if cleaned in {"", "-"}:
        return None
    value = float(cleaned)
    return -value if negative else value


def column(frame, *names):
    lookup = {name.casefold(): name for name in frame.columns}
    for name in names:
        if name.casefold() in lookup:
            return lookup[name.casefold()]
    return None


def first_value(frame, *names):
    found = column(frame, *names)
    if found is None:
        return ""
    for value in frame[found]:
        if str(value).strip():
            return str(value).strip()
    return ""


def broker_for(frame):
    on_row = first_value(frame, "Broker")
    if on_row:
        return on_row
    filename = frame["source_file"].iloc[0]
    controls = source_controls[source_controls["source_file"].eq(filename)]
    labelled = controls[controls["label"].str.casefold().eq("broker")]
    if not labelled.empty and str(labelled.iloc[0]["value"]).strip():
        return str(labelled.iloc[0]["value"]).strip()
    cover = controls[controls["control_type"].eq("cover_sheet") & controls["value"].eq("")]
    if not cover.empty:
        return str(cover.iloc[0]["label"]).strip()
    return ""


def broker_key(name):
    text = re.sub(r"[^A-Z0-9 ]", " ", str(name).upper())
    text = re.sub(r"\b(LTD|LIMITED|PLC|SA|LLP)\b", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def policy_key(reference):
    """Comparable policy key. Does not replace the reference stored on the row."""
    text = str(reference).upper().strip()
    if not text:
        return ""
    text = re.sub(r"[-/\s]*(CAN|ENDT\d*|MTA)$", "", text)
    text = re.sub(r"\s+", "", text)
    year_only = re.fullmatch(r"(\d{4})[/-](\d+)", text)
    if year_only:
        text = f"POL-{year_only.group(1)}-{int(year_only.group(2)):04d}"
    short_pol = re.fullmatch(r"POL(\d{1,4})", text)
    if short_pol:
        text = f"POL-2026-{int(short_pol.group(1)):04d}"
    return re.sub(r"[^A-Z0-9]", "", text)


def name_key(name):
    text = str(name).upper().replace("&", " AND ")
    text = re.sub(r"\b(LTD|LIMITED|PLC|SA|LLP|GROUP)\b", " ", text)
    return re.sub(r"[^A-Z0-9]", "", text)


def txn_group(label):
    return TXN_GROUPS.get(str(label).upper().strip(), "")


def money_fields(frame):
    if column(frame, "GBP Settlement Amount"):
        return column(frame, "GBP Settlement Amount"), "GBP"
    for name in ("Net Settlement", "Amount Due MGA", "Amount Due", "Net Due"):
        found = column(frame, name)
        if found:
            return found, ""
    return None, ""


lines = []
for dataset_name, frame in remittance_details.items():
    amount_column, forced_currency = money_fields(frame)
    policy_column = column(frame, "Policy No", "Policy Reference", "UW Ref", "Policy", "Policy / Risk Ref")
    insured_column = column(frame, "Insured", "Insured Name", "Client", "Assured")
    txn_column = column(frame, "Transaction Type", "Txn Type")
    currency_column = column(frame, "Currency")
    broker = broker_for(frame)
    for row in frame.to_dict("records"):
        amount_text = row.get(amount_column, "") if amount_column else ""
        currency = forced_currency or (row.get(currency_column, "") if currency_column else "") or "GBP"
        policy_reference = row.get(policy_column, "") if policy_column else ""
        lines.append({
            "broker": broker,
            "broker_key": broker_key(broker),
            "source_file": row["source_file"],
            "source_sheet": row["source_sheet"],
            "source_row": row["source_row"],
            "policy_reference": policy_reference,
            "policy_key": policy_key(policy_reference),
            "insured_name": row.get(insured_column, "") if insured_column else "",
            "txn_type": row.get(txn_column, "") if txn_column else "",
            "amount": parse_money(amount_text),
            "currency": currency,
        })

bdx_rows = []
for row in premium_bdx.itertuples(index=False):
    bdx_rows.append({
        "bdx_record_id": row.bdx_record_id,
        "broker": row.broker_name,
        "broker_key": broker_key(row.broker_name),
        "policy_reference": row.policy_reference,
        "policy_key": policy_key(row.policy_reference),
        "insured_name": row.insured_name,
        "txn_type": row.transaction_type,
        "amount": parse_money(row.settlement_amount_gbp),
        "currency": row.settlement_currency,
        "risk_reference": row.risk_reference,
        "used": False,
    })


def numeric_tail(reference):
    digits = re.sub(r"\D", "", str(reference))
    return digits.lstrip("0")


def policy_and_risk_suffix(reference, bdx_row):
    """True when the remittance number is the tail of both the policy and the risk ref."""
    tail = numeric_tail(reference)
    if len(tail) < 3:
        return False
    policy_digits = numeric_tail(bdx_row["policy_reference"])
    risk_digits = numeric_tail(bdx_row["risk_reference"])
    return bool(policy_digits.endswith(tail) and risk_digits.endswith(tail))


def describe_checks(line, match):
    notes = []
    if name_key(line["insured_name"]) == name_key(match["insured_name"]):
        notes.append("insured name agrees")
    else:
        notes.append(f"insured name differs ({line['insured_name']} vs {match['insured_name']})")
    if line["txn_type"] and match["txn_type"] and line["txn_type"].upper() != match["txn_type"].upper():
        if txn_group(line["txn_type"]) and txn_group(line["txn_type"]) == txn_group(match["txn_type"]):
            notes.append(f"transaction label {line['txn_type']} vs {match['txn_type']}")
        else:
            notes.append(f"transaction type {line['txn_type']} vs {match['txn_type']}")
    if line["currency"] != match["currency"]:
        notes.append(f"currency {line['currency']} vs {match['currency']}")
    return "; ".join(notes)


def business_signature(line):
    """Fields that make one remittance line the same item of business as another."""
    return (
        line["broker_key"],
        line["policy_key"],
        name_key(line["insured_name"]),
        line["txn_type"].upper().strip(),
        line["currency"],
        line["amount"],
    )


def candidate_text(row):
    amount = "amount missing" if row["amount"] is None else f"{row['amount']:,.2f} {row['currency']}"
    return f"{row['bdx_record_id']} policy {row['policy_reference']} risk {row['risk_reference']} {row['insured_name']} {amount}"


results = []
seen_business = set()
for line in lines:
    same_broker = [row for row in bdx_rows if row["broker_key"] == line["broker_key"]]
    policy_hits = [row for row in same_broker if line["policy_key"] and row["policy_key"] == line["policy_key"]]
    status = ""
    reason = ""
    reason_code = ""
    match = None
    match_method = ""

    signature = business_signature(line)
    duplicate = bool(line["policy_key"]) and signature in seen_business
    if line["policy_key"]:
        seen_business.add(signature)

    if duplicate:
        status = "duplicate remittance line"
        reason_code = "DUPLICATE_LINE"
        reason = (
            "The broker, policy reference, insured name, transaction type, currency, and amount "
            "repeat an earlier remittance line. The BDX record was not used a second time."
        )
    elif len(policy_hits) == 1:
        match = policy_hits[0]
        match_method = "policy reference"
        if match["used"]:
            status = "manual review"
            reason_code = "MANUAL_REVIEW"
            reason = (
                f"{match['bdx_record_id']} is already linked to another remittance line, "
                "and this line is not a repeat of that line's business fields."
            )
            match = None
            match_method = ""
        else:
            match["used"] = True
    elif len(policy_hits) > 1:
        status = "unresolved"
        reason_code = "AMBIGUOUS_REFERENCE"
        ids = ", ".join(row["bdx_record_id"] for row in policy_hits)
        reason = f"More than one BDX record shares this policy reference ({ids}). Left unresolved."
    elif not line["policy_key"]:
        unused = [row for row in same_broker if not row["used"]]
        exact_names = [
            row for row in unused
            if name_key(line["insured_name"]) and name_key(row["insured_name"]) == name_key(line["insured_name"])
        ]
        loose_names = [
            row for row in unused
            if name_key(line["insured_name"]) and name_key(line["insured_name"]) in name_key(row["insured_name"])
        ]
        candidates = exact_names or loose_names
        status = "unresolved"
        if len(candidates) > 1:
            reason_code = "AMBIGUOUS_REFERENCE"
            listed = "; ".join(candidate_text(row) for row in candidates)
            reason = (
                "No policy reference. These BDX records are unconfirmed candidates, not ruled out: "
                f"{listed}."
            )
        elif len(candidates) == 1:
            reason_code = "MANUAL_REVIEW"
            reason = (
                f"No policy reference. Unconfirmed candidate {candidate_text(candidates[0])}. "
                "A similar insured name alone is not enough to assign it."
            )
        else:
            reason_code = "NO_BDX_MATCH"
            reason = "No policy reference, and no BDX insured name on this broker is a candidate."
    else:
        suffix_hits = [row for row in same_broker if not row["used"] and policy_and_risk_suffix(line["policy_reference"], row)]
        supported = [
            row for row in suffix_hits
            if name_key(line["insured_name"]) == name_key(row["insured_name"])
            and line["currency"] == row["currency"]
            and line["amount"] is not None
            and row["amount"] is not None
            and abs(line["amount"] - row["amount"]) <= PENNY_TOLERANCE
        ]
        if len(supported) == 1:
            match = supported[0]
            match["used"] = True
            match_method = "policy and risk reference suffix"
        elif len(suffix_hits) >= 1:
            status = "unresolved"
            reason_code = "MANUAL_REVIEW"
            listed = "; ".join(candidate_text(row) for row in suffix_hits)
            reason = (
                f"{line['policy_reference']} is not a BDX policy reference. "
                f"Possible BDX candidate(s) for manual review, not assigned: {listed}."
            )
        else:
            status = "unresolved"
            reason_code = "NO_BDX_MATCH"
            reason = f"No BDX policy reference matches {line['policy_reference']} for this broker."

    variance = None
    if match is not None:
        checks = describe_checks(line, match)
        if line["amount"] is None or match["amount"] is None:
            status = "missing amount"
            reason_code = "MISSING_AMOUNT"
            reason = (
                f"Linked to {match['bdx_record_id']} by {match_method}. {checks}. "
                "An amount is missing, so the amounts were not compared."
            )
            match_method = ""
            # Keep the link only when we could compare. A missing amount is not a confirmed amount match.
            # The record was marked used above; release it if we cannot confirm the amount.
            match["used"] = False
            bdx_id = match["bdx_record_id"]
            bdx_policy = match["policy_reference"]
            match = None
            reason = (
                f"Possible link to {bdx_id} ({bdx_policy}) by reference, but an amount is missing. "
                "Not treated as a confirmed match."
            )
            reason_code = "MISSING_AMOUNT"
            status = "missing amount"
        elif line["currency"] != match["currency"]:
            status = "currency mismatch"
            reason_code = "CURRENCY_MISMATCH"
            bdx_id = match["bdx_record_id"]
            match["used"] = False
            match = None
            variance = None
            reason = (
                f"Reference points at {bdx_id}, but the currencies differ. "
                "Amounts were not compared and the record was not confirmed."
            )
        else:
            variance = round(line["amount"] - match["amount"], 2)
            if match_method == "policy and risk reference suffix":
                how = (
                    f"{line['policy_reference']} is not the BDX policy reference. "
                    f"With punctuation removed, its numeric tail is the trailing part of both "
                    f"policy {match['policy_reference']} and risk {match['risk_reference']} "
                    f"on {match['bdx_record_id']}, and of no other record for this broker. "
                    f"{checks}."
                )
            else:
                how = f"Policy reference matches {match['bdx_record_id']}. {checks}."
            if abs(variance) <= PENNY_TOLERANCE:
                status = "matched"
                reason_code = "MATCHED"
                reason = how
            else:
                status = "amount difference"
                reason_code = "AMOUNT_VARIANCE"
                reason = (
                    f"{how} Remittance {line['amount']:,.2f} vs BDX {match['amount']:,.2f} "
                    f"({line['currency']}; variance {variance:,.2f})."
                )

    results.append({
        "broker": line["broker"],
        "source_file": line["source_file"],
        "source_sheet": line["source_sheet"],
        "source_row": line["source_row"],
        "remittance_policy_reference": line["policy_reference"],
        "insured_name": line["insured_name"],
        "bdx_record_id": match["bdx_record_id"] if match else "",
        "bdx_policy_reference": match["policy_reference"] if match else "",
        "match_method": match_method if match else "",
        "remittance_amount": line["amount"],
        "bdx_amount": match["amount"] if match else None,
        "currency": line["currency"],
        "amount_variance": variance,
        "status": status,
        "reason_code": reason_code,
        "reason": reason,
    })

remittance_to_bdx = pd.DataFrame(results)
bdx_without_remittance = pd.DataFrame([
    {
        "bdx_record_id": row["bdx_record_id"],
        "broker": row["broker"],
        "bdx_policy_reference": row["policy_reference"],
        "insured_name": row["insured_name"],
        "transaction_type": row["txn_type"],
        "settlement_amount": row["amount"],
        "currency": row["currency"],
        "review_note": (
            "Unconfirmed candidate for the Harbour line with no policy reference; not a confirmed match."
            if row["bdx_record_id"] in {"BDX-0019", "BDX-0020"}
            else "No remittance line confirms this BDX record."
        ),
    }
    for row in bdx_rows
    if not row["used"]
])

pd.set_option("display.max_colwidth", 140)
pd.set_option("display.max_rows", 40)
display(remittance_to_bdx)
print(f"Rounding tolerance: {PENNY_TOLERANCE:.2f}. Variance is remittance amount minus BDX settlement amount.")
print("BDX records with no remittance match:")
display(bdx_without_remittance)

,broker,source_file,source_sheet,source_row,remittance_policy_reference,insured_name,bdx_record_id,bdx_policy_reference,match_method,remittance_amount,bdx_amount,currency,amount_variance,status,reason_code,reason
0,Apex Risk Partners Ltd,apex_risk_partners_remittance_apr_2026.csv,,6,2026/0001,Sterling Construction Limited,BDX-0001,POL-2026-0001,policy reference,8500.0,8500.0,GBP,0.0,matched,MATCHED,Policy reference matches BDX-0001. insured name agrees; transaction label NB vs New Business.
1,Apex Risk Partners Ltd,apex_risk_partners_remittance_apr_2026.csv,,7,POL0002,A.B.C. Technology Ltd,BDX-0002,POL-2026-0002,policy reference,6375.0,6375.0,GBP,0.0,matched,MATCHED,Policy reference matches BDX-0002. insured name agrees; transaction label NB vs New Business.
2,Apex Risk Partners Ltd,apex_risk_partners_remittance_apr_2026.csv,,8,POL-2026-0003-CAN,Blue Harbour Estates Limited,BDX-0003,POL-2026-0003,policy reference,-1020.0,-1020.0,GBP,0.0,matched,MATCHED,Policy reference matches BDX-0003. insured name agrees; transaction label CAN vs Cancellation.
3,Apex Risk Partners Ltd,apex_risk_partners_remittance_apr_2026.csv,,9,POL-2026-0004-ENDT01,Greenfield Farms Ltd,BDX-0004,POL-2026-0004,policy reference,1912.5,1912.5,GBP,0.0,matched,MATCHED,Policy reference matches BDX-0004. insured name agrees; transaction label MTA vs Endorsement.
4,Apex Risk Partners Ltd,apex_risk_partners_remittance_apr_2026.csv,,10,POL-2026-0005,Northshore Energy Services Ltd,BDX-0005,POL-2026-0005,policy reference,8250.0,8500.0,GBP,-250.0,amount difference,AMOUNT_VARIANCE,"Policy reference matches BDX-0005. insured name agrees; transaction label NB vs New Business. Remittance 8,250.00 vs BDX 8,500.00 (GBP; ..."
5,Apex Risk Partners Ltd,apex_risk_partners_remittance_apr_2026.csv,,11,POL-2026-0006,Thames Logistics Limited,BDX-0006,POL-2026-0006,policy reference,4250.0,4250.0,GBP,0.0,matched,MATCHED,Policy reference matches BDX-0006. insured name agrees; transaction label NB vs New Business.
6,Apex Risk Partners Ltd,apex_risk_partners_remittance_apr_2026.csv,,12,POL-2026-0006,Thames Logistics Limited,,,,4250.0,NaN,GBP,NaN,duplicate remittance line,DUPLICATE_LINE,"The broker, policy reference, insured name, transaction type, currency, and amount repeat an earlier remittance line. The BDX record was..."
7,Continental Coverholders SA,continental_coverholders_apr_2026.xlsx,EUR Settlement,6,POL-2026-0401,Riviera Foods SARL,BDX-0015,POL-2026-0401,policy reference,8600.0,8600.0,GBP,0.0,matched,MATCHED,Policy reference matches BDX-0015. insured name agrees; transaction label NB vs New Business.
8,Continental Coverholders SA,continental_coverholders_apr_2026.xlsx,EUR Settlement,7,POL-2026-0402,Alpine Components GmbH,BDX-0016,POL-2026-0402,policy reference,4300.0,4300.0,GBP,0.0,matched,MATCHED,Policy reference matches BDX-0016. insured name agrees; transaction label NB vs New Business.
9,Continental Coverholders SA,continental_coverholders_apr_2026.xlsx,EUR Settlement,8,POL-2026-0403,Nordic Fleet AB,BDX-0017,POL-2026-0403,policy reference,-1032.0,-1032.0,GBP,0.0,matched,MATCHED,Policy reference matches BDX-0017. insured name agrees; transaction label CAN vs Cancellation.


Rounding tolerance: 0.01. Variance is remittance amount minus BDX settlement amount.
BDX records with no remittance match:


,bdx_record_id,broker,bdx_policy_reference,insured_name,transaction_type,settlement_amount,currency,review_note
0,BDX-0019,Harbour Specialty Ltd,POL-2026-0302,Orange Retail Ltd,New Business,7200.0,GBP,Unconfirmed candidate for the Harbour line with no policy reference; not a confirmed match.
1,BDX-0020,Harbour Specialty Ltd,POL-2026-0303,Orange Retail Group Ltd,New Business,7200.0,GBP,Unconfirmed candidate for the Harbour line with no policy reference; not a confirmed match.
2,BDX-0022,Apex Risk Partners Ltd,POL-2026-0007,Meadow Labs Ltd,New Business,12800.0,GBP,No remittance line confirms this BDX record.


## Summary And Exceptions

The two stages stay separate. Bank cash is not added to remittance or BDX amounts, and figures in different currencies are not combined.

**Tolerance.** A variance of at most **0.01** in the same currency is treated as matched. Larger differences are shown as the actual variance. The tolerance is one penny of the compared currency; every compared amount in this pack is GBP.


In [18]:
# Bank cash and statement totals stay in their own currency. They are not added to BDX amounts.

linked_receipts = bank_receipt_lines[bank_receipt_lines["statement_reference"].ne("")]
unidentified_receipts = bank_receipt_lines[bank_receipt_lines["statement_reference"].eq("")]

cash_rows = []
for currency, linked in linked_receipts.groupby("currency"):
    unidentified = unidentified_receipts[unidentified_receipts["currency"].eq(currency)]
    cash_rows.append({
        "currency": currency,
        "linked_to_a_statement": round(float(linked["amount"].sum()), 2),
        "linked_receipt_count": int(len(linked)),
        "unidentified": round(float(unidentified["amount"].sum()), 2) if not unidentified.empty else 0.0,
        "unidentified_receipt_count": int(len(unidentified)),
    })
bank_cash_position = pd.DataFrame(cash_rows)

statement_position = bank_to_remittance[bank_to_remittance["statement_reference"].ne("")][
    [
        "broker", "statement_reference", "currency", "statement_total",
        "bank_amount_received", "variance", "reason_code", "bank_transaction_ids",
    ]
].copy()
def payment_position(row):
    if row.reason_code == "MATCHED":
        return "fully paid"
    if row.reason_code == "NO_IDENTIFIED_RECEIPT":
        return "no identified receipt"
    if row.reason_code == "AMOUNT_VARIANCE":
        return "short paid" if row.variance < 0 else "overpaid"
    if row.reason_code == "CURRENCY_MISMATCH":
        return "currency mismatch — amounts not compared"
    if row.reason_code == "MISSING_AMOUNT":
        return "missing amount — amounts not compared"
    return row.reason_code

statement_position["payment_position"] = statement_position.apply(payment_position, axis=1)

line_counts = (
    remittance_to_bdx["reason_code"]
    .value_counts()
    .rename_axis("reason_code")
    .reset_index(name="remittance_lines")
)

print(
    f"Tolerance: {ROUNDING_TOLERANCE:.2f} in the same currency. "
    "A statement is fully paid only when reason_code is MATCHED."
)
print("Bank fee debits are excluded from the cash totals:")
display(excluded_debits[["transaction_id", "currency", "debit", "narrative"]])
display(bank_cash_position)
display(statement_position)
display(line_counts)
print("BDX records with no confirmed remittance match:")
display(bdx_without_remittance)


Tolerance: 0.01 in the same currency. A statement is fully paid only when reason_code is MATCHED.
Bank fee debits are excluded from the cash totals:


,transaction_id,currency,debit,narrative
6,BANK-0007,GBP,30.0,MONTHLY ACCOUNT CHARGES


,currency,linked_to_a_statement,linked_receipt_count,unidentified,unidentified_receipt_count
0,GBP,294375.3,5,28750.0,1


,broker,statement_reference,currency,statement_total,bank_amount_received,variance,reason_code,bank_transaction_ids,payment_position
0,Apex Risk Partners Ltd,APX-APR-2026,GBP,32517.5,32517.5,0.0,MATCHED,BANK-0001,fully paid
1,Continental Coverholders SA,CON-APR-2026,GBP,11868.0,11867.8,-0.2,AMOUNT_VARIANCE,BANK-0005,short paid
2,Harbour Specialty Ltd,HRB-APR-2026,GBP,36500.0,0.0,NaN,NO_IDENTIFIED_RECEIPT,,no identified receipt
3,Northshore Broking Ltd,REM-0426-NSB,GBP,50000.0,49990.0,-10.0,AMOUNT_VARIANCE,BANK-0002,short paid
4,Pioneer Wholesale Brokers,PIO-MAY26,GBP,200000.0,200000.0,0.0,MATCHED,"BANK-0003, BANK-0004",fully paid


,reason_code,remittance_lines
0,MATCHED,18
1,AMOUNT_VARIANCE,1
2,DUPLICATE_LINE,1
3,AMBIGUOUS_REFERENCE,1


BDX records with no confirmed remittance match:


,bdx_record_id,broker,bdx_policy_reference,insured_name,transaction_type,settlement_amount,currency,review_note
0,BDX-0019,Harbour Specialty Ltd,POL-2026-0302,Orange Retail Ltd,New Business,7200.0,GBP,Unconfirmed candidate for the Harbour line with no policy reference; not a confirmed match.
1,BDX-0020,Harbour Specialty Ltd,POL-2026-0303,Orange Retail Group Ltd,New Business,7200.0,GBP,Unconfirmed candidate for the Harbour line with no policy reference; not a confirmed match.
2,BDX-0022,Apex Risk Partners Ltd,POL-2026-0007,Meadow Labs Ltd,New Business,12800.0,GBP,No remittance line confirms this BDX record.


## Written Notes

**Northshore, £10 short.** Bank credit `BANK-0002` is linked to remittance `REM-0426-NSB` by the name Northshore and the remittance id. The statement total is GBP 50,000.00 and the credit is GBP 49,990.00, so the variance is GBP -10.00. That is above the 0.01 tolerance, so the statement is `AMOUNT_VARIANCE`, not matched. The narrative says `OUR CHG GBP10 NET`, which accounts for the £10. The four remittance lines themselves match the BDX to the penny. Follow-up: confirm the £10 is the bank’s CHAPS charge and whether the broker or the MGA should bear it.

**Continental, 20p after FX.** `BANK-0005` names Continental Coverholders and is GBP 11,867.80. The statement total is GBP 11,868.00, variance GBP -0.20, so the statement is `AMOUNT_VARIANCE`. The narrative records an EUR settlement at FX 0.8600. The three remittance lines match the BDX GBP settlement amounts exactly (GBP 8,600.00, 4,300.00, and -1,032.00). The 20p sits between the bank and the statement total, not between the remittance and the BDX. Follow-up: ask whether the FX rounding on the payment should be written off or claimed.

**Pioneer, two receipts, one statement.** `BANK-0003` (GBP 150,000.00 on 4 May) and `BANK-0004` (GBP 50,000.00 on 5 May) both name Pioneer and quote `PIO-MAY26`. Together they equal the GBP 200,000.00 statement total, variance 0.00, so the statement is `MATCHED`. The four remittance lines also match the BDX. No follow-up on the split payment itself.

**Unknown GBP 28,750 credit.** `BANK-0006` is a GBP 28,750.00 credit from `UNKNOWN BROKER SERVICES`, narrative `UBS CLIENT MONEY MAY PREMIUMS`. No broker name or statement reference on the remittances appears in those fields, so it is `UNMATCHED_BANK_CREDIT` and is not allocated by amount. It is the only unidentified credit. Follow-up: identify the paying broker before it is applied to a statement.

**Thames Logistics repeated line.** Apex lists `POL-2026-0006`, Thames Logistics Limited, GBP 4,250.00 twice, on source rows 11 and 12. The insured name, transaction type, currency, and amount are the same, so the second line is `DUPLICATE_LINE`. `BDX-0006` is linked only to the first line. A repeated policy reference would not be flagged this way if the business fields differed. Follow-up: confirm with Apex whether the second line was sent twice in error. The statement total of GBP 32,517.50 includes both lines and still equals `BANK-0001`, so the cash agrees with the statement as sent.

**Apex Northshore Energy, GBP 250.** Remittance line `POL-2026-0005` matches `BDX-0005` on policy reference, broker, and insured name. The remittance net is GBP 8,250.00 and the BDX settlement is GBP 8,500.00, variance GBP -250.00, so the line is `AMOUNT_VARIANCE`. The statement cash still matches the remittance total, so the difference is between the remittance and the BDX, not a missing bank receipt. Follow-up: ask Apex why the net due is GBP 250 below the BDX.

**Harbour `H-0301`.** Line `HRB-1` quotes `H-0301`, not `POL-2026-0301`. For Harbour, the numeric tail 301 is the trailing part of both policy `POL-2026-0301` and risk `RISK-HR-301` on `BDX-0018`, and of no other Harbour record. The insured name agrees once Limited/Ltd is ignored, and both amounts are GBP 12,000.00. That is a unique link, recorded as a policy-and-risk-suffix match, status `MATCHED`. No further reference follow-up is needed unless the broker’s coding of `H-0301` is disputed.

**Harbour Orange Retail, no policy reference.** Line `HRB-2` has a blank policy reference, insured `Orange Retail`, and net GBP 7,200.00. Two unused Harbour BDX records fit that name and amount: `BDX-0019` `POL-2026-0302` Orange Retail Ltd and `BDX-0020` `POL-2026-0303` Orange Retail Group Ltd, each GBP 7,200.00. The line is `AMBIGUOUS_REFERENCE`. Those records are unconfirmed candidates, not ruled out. Follow-up: ask Harbour which policy this line pays.

**Harbour statement with no identified payment.** Statement `HRB-APR-2026` totals GBP 36,500.00. No bank credit names Harbour or quotes that statement reference, so the statement is `NO_IDENTIFIED_RECEIPT`. The GBP 28,750 unknown credit is not applied to it. The Canal Medical line matches `BDX-0021`, and `H-0301` matches `BDX-0018`; only the Orange Retail line is unresolved. Follow-up: locate the Harbour payment, which is not in this bank file.
